# Ad Click Prediction Case Study ? Corrected Version

This notebook implements the reviewer feedback: leakage-safe historical CTR features, explicit baselines, feature ablation, corrected model-selection rationale, cautious performance language, individual feature importance, and clearer business mapping.

## 1. Executive Summary

The solution predicts rare ad clicks using temporal, contextual, and historical CTR features. Because validation performance is moderate rather than high, predictions should be used as a ranking and targeting aid, with threshold tuning and live monitoring.

## 2. Business Problem and ML Objective

The ad-tech objective is to reduce wasted impressions and prioritize users with higher click propensity. The ML task is imbalanced binary classification, evaluated with ROC-AUC, PR-AUC, F1, precision, recall, and confusion-matrix trade-offs.

## 3. Dataset Understanding

The training data contains impression-level records with user, product, campaign, webpage, demographic, and timestamp fields. The target is `is_click`.

## 4. Validation Strategy

A random stratified split is not used as the primary validation strategy because ad serving is temporal. The latest date is held out to approximate future serving and avoid future behavior leaking into model development.

## 5. Leakage-Safe Historical Feature Engineering

Historical CTR features are computed cumulatively for training rows. Each row only uses impressions that occurred before the current impression. Validation rows use only development-history maps; test rows use all training-history maps.

## 6. Baselines, Ablation, Models, and Evaluation

The notebook compares always-negative and global-CTR baselines, feature-set ablations, class-weighted models, threshold tuning, ROC/PR curves, calibration, and feature importance.

## 7. Business Questions and Recommendations

The analysis connects model evidence to weekend bidding, product performance, personalization, SMOTE trade-offs, inventory planning signals, user profiles, and operational monitoring.

In [1]:

# Ad Click Prediction - corrected leakage-safe case study
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, precision_recall_curve, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

RANDOM_STATE = 42
DATA_DIR = Path('assets/data')
IMAGE_DIR = Path('assets/images')
OUTPUT_DIR = Path('outputs')
REPORT_DIR = Path('reports')
for d in [IMAGE_DIR, OUTPUT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'Ad_click_prediction_train (1).csv'
TEST_PATH = DATA_DIR / 'Ad_Click_prediciton_test.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train['DateTime'] = pd.to_datetime(train['DateTime'])
test['DateTime'] = pd.to_datetime(test['DateTime'])
print('Train:', train.shape, 'Test:', test.shape)
print('Click rate:', round(train['is_click'].mean() * 100, 2), '%')

def add_time_features(df):
    out = df.copy()
    out['hour'] = out['DateTime'].dt.hour
    out['day'] = out['DateTime'].dt.day
    out['dayofweek'] = out['DateTime'].dt.dayofweek
    out['is_weekend'] = (out['dayofweek'] >= 5).astype(int)
    out['is_night'] = out['hour'].between(0, 5).astype(int)
    out['is_business_hour'] = out['hour'].between(9, 18).astype(int)
    return out

def add_interaction_keys(df):
    out = df.copy()
    out['user_product'] = out['user_id'].astype(str) + '_' + out['product'].astype(str)
    out['campaign_product'] = out['campaign_id'].astype(str) + '_' + out['product'].astype(str)
    out['webpage_product'] = out['webpage_id'].astype(str) + '_' + out['product'].astype(str)
    return out

def add_cumulative_history_features(df, cols, target='is_click', smoothing=30):
    """Leakage-safe historical CTR/counts for training rows.
    Each row only sees impressions/clicks that occurred before itself.
    """
    out = df.sort_values(['DateTime', 'session_id']).copy()
    prior = out[target].mean()
    for col in cols:
        previous_count = out.groupby(col).cumcount()
        previous_clicks = out.groupby(col)[target].cumsum() - out[target]
        out[f'{col}_hist_impressions'] = previous_count.astype(float)
        out[f'{col}_hist_clicks'] = previous_clicks.astype(float)
        out[f'{col}_hist_ctr'] = (previous_clicks + smoothing * prior) / (previous_count + smoothing)
    return out.sort_index(), prior

def build_history_maps(df, cols, target='is_click', smoothing=30):
    prior = df[target].mean()
    maps = {}
    for col in cols:
        stats = df.groupby(col)[target].agg(['sum', 'count'])
        stats['hist_ctr'] = (stats['sum'] + smoothing * prior) / (stats['count'] + smoothing)
        maps[col] = {
            'ctr': stats['hist_ctr'].to_dict(),
            'clicks': stats['sum'].to_dict(),
            'impressions': stats['count'].to_dict(),
        }
    return maps, prior

def apply_history_maps(df, maps, prior, cols):
    out = df.copy()
    for col in cols:
        out[f'{col}_hist_impressions'] = out[col].map(maps[col]['impressions']).fillna(0).astype(float)
        out[f'{col}_hist_clicks'] = out[col].map(maps[col]['clicks']).fillna(0).astype(float)
        out[f'{col}_hist_ctr'] = out[col].map(maps[col]['ctr']).fillna(prior).astype(float)
    return out

train_fe = add_interaction_keys(add_time_features(train))
test_fe = add_interaction_keys(add_time_features(test))
cutoff = train_fe['DateTime'].max().normalize()
dev_raw = train_fe[train_fe['DateTime'] < cutoff].copy()
valid_raw = train_fe[train_fe['DateTime'] >= cutoff].copy()
print('Temporal holdout cutoff:', cutoff.date())
print('Dev:', dev_raw.shape, 'Valid:', valid_raw.shape, 'Valid click rate:', round(valid_raw.is_click.mean(), 4))
print('Random stratified split is not primary because future ad-serving behavior should not leak backward into training.')

history_cols = [
    'product', 'campaign_id', 'webpage_id', 'product_category_1', 'product_category_2',
    'user_group_id', 'age_level', 'city_development_index', 'user_product',
    'campaign_product', 'webpage_product'
]
interaction_cols = ['user_product', 'campaign_product', 'webpage_product']

dev_hist, dev_prior = add_cumulative_history_features(dev_raw, history_cols)
dev_maps, _ = build_history_maps(dev_raw, history_cols)
valid_hist = apply_history_maps(valid_raw, dev_maps, dev_prior, history_cols)

# Keep model iteration practical while validating on the full temporal holdout.
model_train_idx, _ = train_test_split(
    np.arange(len(dev_hist)),
    train_size=min(150000, len(dev_hist)),
    stratify=dev_hist['is_click'],
    random_state=RANDOM_STATE,
)
dev_model = dev_hist.iloc[model_train_idx].copy()
print('Model training sample:', dev_model.shape, 'Full validation:', valid_hist.shape)

baseline_features = [
    'product','gender','campaign_id','webpage_id','product_category_1','product_category_2',
    'user_group_id','age_level','user_depth','city_development_index','var_1',
    'hour','dayofweek','is_weekend','is_night','is_business_hour'
]
interaction_count_features = [f'{c}_hist_impressions' for c in interaction_cols] + [f'{c}_hist_clicks' for c in interaction_cols]
historical_ctr_features = [f'{c}_hist_ctr' for c in history_cols]
all_numeric_features = interaction_count_features + historical_ctr_features
full_features = baseline_features + all_numeric_features
cat_features = baseline_features

def make_preprocess(numeric_features):
    return ColumnTransformer([
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=20)),
        ]), cat_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
    ])

def evaluate_probs(y_true, probs, name, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {
        'model': name, 'threshold': float(threshold),
        'roc_auc': roc_auc_score(y_true, probs),
        'pr_auc': average_precision_score(y_true, probs),
        'f1': f1_score(y_true, preds, zero_division=0),
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds, zero_division=0),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)
    }

def best_threshold_by_f1(y_true, probs):
    thresholds = np.linspace(0.03, 0.5, 80)
    f1s = [f1_score(y_true, probs >= t, zero_division=0) for t in thresholds]
    return float(thresholds[int(np.argmax(f1s))])

def fit_eval_model(name, estimator, features, numeric_features):
    pipe = Pipeline([('prep', make_preprocess(numeric_features)), ('model', estimator)])
    pipe.fit(dev_model[features], dev_model['is_click'])
    probs = pipe.predict_proba(valid_hist[features])[:, 1]
    threshold = best_threshold_by_f1(valid_hist['is_click'], probs)
    return pipe, probs, [evaluate_probs(valid_hist['is_click'], probs, name, 0.5), evaluate_probs(valid_hist['is_click'], probs, name + ' - tuned threshold', threshold)]

# Proper baselines
baseline_rows = []
y_valid = valid_hist['is_click']
global_ctr = dev_hist['is_click'].mean()
baseline_rows.append(evaluate_probs(y_valid, np.full(len(y_valid), global_ctr), 'Global CTR probability baseline', global_ctr))
baseline_rows.append({
    'model': 'Always non-click baseline', 'threshold': 0.5, 'roc_auc': 0.5,
    'pr_auc': y_valid.mean(), 'f1': 0.0, 'precision': 0.0, 'recall': 0.0,
    'tn': int((y_valid == 0).sum()), 'fp': 0, 'fn': int((y_valid == 1).sum()), 'tp': 0
})

# Feature ablation using the same balanced logistic estimator for clean comparison.
ablation_specs = [
    ('Baseline raw + temporal', baseline_features, []),
    ('+ Interaction history counts', baseline_features + interaction_count_features, interaction_count_features),
    ('+ Leakage-safe historical CTR', full_features, all_numeric_features),
]
ablation_rows = []
for label, feats, nums in ablation_specs:
    estimator = LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
    _, probs, rows = fit_eval_model(label, estimator, feats, nums)
    tuned = [r for r in rows if r['model'].endswith('tuned threshold')][0]
    ablation_rows.append(tuned)
ablation = pd.DataFrame(ablation_rows)
display(pd.DataFrame(baseline_rows))
display(ablation)

models = {
    'Logistic Regression - balanced': LogisticRegression(max_iter=500, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
    'Random Forest - balanced': RandomForestClassifier(n_estimators=80, max_depth=10, min_samples_leaf=50, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE),
    'XGBoost - weighted': XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.08, subsample=0.85, colsample_bytree=0.85, eval_metric='logloss', scale_pos_weight=(dev_model['is_click'].eq(0).sum()/dev_model['is_click'].eq(1).sum()), random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist')
}
results = baseline_rows.copy()
fitted = {}
probs_by_model = {}
for name, estimator in models.items():
    pipe, probs, rows = fit_eval_model(name, estimator, full_features, all_numeric_features)
    fitted[name] = pipe
    probs_by_model[name] = probs
    results.extend(rows)
metrics = pd.DataFrame(results).sort_values(['f1', 'roc_auc'], ascending=False)
display(metrics)

best_tuned = metrics[metrics['model'].str.endswith(' - tuned threshold')].iloc[0]
best_name = best_tuned['model'].replace(' - tuned threshold', '')
best_threshold = float(best_tuned['threshold'])
best_pipe = fitted[best_name]
best_probs = probs_by_model[best_name]
print('Selected:', best_name, 'threshold:', round(best_threshold, 3))
print(classification_report(y_valid, best_probs >= best_threshold, zero_division=0))
print('Selection rationale: highest F1 among evaluated tuned-threshold classifiers; PR-AUC and ROC-AUC leaders are reported separately.')

# Corrected imbalance discussion: SMOTE/SMOTENC is not run because imblearn is unavailable.
# This avoids the invalid previous approach of interpolating one-hot encoded categorical values.
try:
    import imblearn  # noqa
    smote_note = 'imblearn is available; SMOTENC can be added before one-hot encoding in a future experiment.'
except Exception:
    smote_note = 'SMOTENC was not executed because imblearn is not installed. The notebook avoids invalid interpolation of one-hot categorical features.'
smote_comparison = {
    'method': 'Class weighting selected over SMOTE',
    'note': smote_note,
    'reason': 'Class weighting improves rare-click sensitivity without generating synthetic categorical records or increasing the training matrix size.'
}
print(smote_comparison)

# Business statistics
weekend = train_fe.groupby('is_weekend')['is_click'].agg(['mean','sum','count']).reset_index()
weekday_ctr = float(weekend.loc[weekend['is_weekend'].eq(0), 'mean'].iloc[0])
weekend_ctr = float(weekend.loc[weekend['is_weekend'].eq(1), 'mean'].iloc[0])
weekend_uplift_pp = (weekend_ctr - weekday_ctr) * 100
weekend_relative_uplift = (weekend_ctr / weekday_ctr - 1) * 100
z_stat, p_value = proportions_ztest(
    weekend.sort_values('is_weekend')['sum'].to_numpy(),
    weekend.sort_values('is_weekend')['count'].to_numpy()
)
print(f'Weekend CTR: {weekend_ctr:.2%}; Weekday CTR: {weekday_ctr:.2%}; uplift: {weekend_uplift_pp:.2f} pp / {weekend_relative_uplift:.1f}% relative; p-value={p_value:.4g}')

product = train_fe.groupby('product')['is_click'].agg(clicks='sum', impressions='count', ctr='mean').reset_index()
product['expected_clicks_per_100k_impressions'] = product['ctr'] * 100000
product = product.sort_values('ctr', ascending=False)
display(product)

profile = train_fe.groupby(['gender','age_level','city_development_index'])['is_click'].agg(clicks='sum', impressions='count', ctr='mean').reset_index()
profile = profile[profile['impressions'] >= 500].copy()
ci_low, ci_high = proportion_confint(profile['clicks'], profile['impressions'], method='wilson')
profile['ctr_ci_low'] = ci_low
profile['ctr_ci_high'] = ci_high
profile = profile.sort_values('ctr', ascending=False).head(10)
display(profile)

# Top individual feature importance; keep exact names instead of broad split('_')[0] grouping.
feature_names = best_pipe.named_steps['prep'].get_feature_names_out()
importances = getattr(best_pipe.named_steps['model'], 'feature_importances_', None)
if importances is not None:
    feature_importance = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(15)
else:
    sample_n = min(15000, len(valid_hist))
    sample = valid_hist.sample(sample_n, random_state=RANDOM_STATE)
    perm = permutation_importance(best_pipe, sample[full_features], sample['is_click'], scoring='average_precision', n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
    feature_importance = pd.DataFrame({'feature': full_features, 'importance': perm.importances_mean}).sort_values('importance', ascending=False).head(15)
display(feature_importance)

# Curves and charts
plt.figure(figsize=(6,4)); train['is_click'].value_counts().sort_index().plot(kind='bar', color=['#64748b','#f59e0b']); plt.xticks([0,1], ['No Click','Click'], rotation=0); plt.title('Target Distribution'); plt.ylabel('Impressions'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'target_distribution.png', dpi=150); plt.show()

plt.figure(figsize=(8,4)); product.sort_values('ctr').plot(kind='barh', x='product', y='ctr', legend=False, ax=plt.gca(), color='#2f6f9f'); plt.title('Product CTR'); plt.xlabel('CTR'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'product_ctr.png', dpi=150); plt.show()

plot_metrics = metrics[metrics['model'].str.endswith('tuned threshold')].sort_values('f1')
plt.figure(figsize=(8,4)); plot_metrics.plot(kind='barh', x='model', y='f1', legend=False, ax=plt.gca(), color='#f59e0b'); plt.title('Tuned-threshold F1 by Model'); plt.xlabel('F1'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'model_f1.png', dpi=150); plt.show()

plt.figure(figsize=(8,4)); ablation.sort_values('f1').plot(kind='barh', x='model', y='f1', legend=False, ax=plt.gca(), color='#0f766e'); plt.title('Feature Ablation: F1 Lift'); plt.xlabel('F1'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'feature_ablation.png', dpi=150); plt.show()

plt.figure(figsize=(8,4)); feature_importance.sort_values('importance').plot(kind='barh', x='feature', y='importance', legend=False, ax=plt.gca(), color='#14b8a6'); plt.title('Top Individual Feature Importances'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'feature_importance.png', dpi=150); plt.show()

precision, recall, _ = precision_recall_curve(y_valid, best_probs)
plt.figure(figsize=(6,4)); plt.plot(recall, precision, color='#b91c1c'); plt.title('Precision-Recall Curve'); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'pr_curve.png', dpi=150); plt.show()

fpr, tpr, _ = roc_curve(y_valid, best_probs)
plt.figure(figsize=(6,4)); plt.plot(fpr, tpr, color='#2563eb'); plt.plot([0,1],[0,1],'--',color='#94a3b8'); plt.title('ROC Curve'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'roc_curve.png', dpi=150); plt.show()

prob_true, prob_pred = calibration_curve(y_valid, best_probs, n_bins=10, strategy='quantile')
plt.figure(figsize=(6,4)); plt.plot(prob_pred, prob_true, marker='o', color='#7c3aed'); plt.plot([0,1],[0,1],'--',color='#94a3b8'); plt.title('Calibration Plot'); plt.xlabel('Mean predicted probability'); plt.ylabel('Observed click rate'); plt.tight_layout(); plt.savefig(IMAGE_DIR/'calibration_plot.png', dpi=150); plt.show()

business_mapping = pd.DataFrame([
    ['High product CTR', 'Increase exposure and reserve more inventory'],
    ['High webpage CTR', 'Prioritize placement and monitor page quality'],
    ['High user-product historical CTR', 'Personalize ad selection and bid higher'],
    ['Weekend uplift', 'Test weekend bid modifier before permanent adoption'],
    ['High predicted probability', 'Prioritize impression under budget constraints'],
    ['Low CTR product', 'Refresh creative or reduce allocation'],
    ['Model drift', 'Retrain and recalibrate thresholds'],
], columns=['Model insight', 'Business action'])
display(business_mapping)

# Final train/test feature construction: training rows are cumulative; test sees all train history.
full_train_hist, full_prior = add_cumulative_history_features(train_fe, history_cols)
full_maps, _ = build_history_maps(train_fe, history_cols)
test_final = apply_history_maps(test_fe, full_maps, full_prior, history_cols)
final_pipe = Pipeline([('prep', make_preprocess(all_numeric_features)), ('model', models[best_name])])
# Final fit uses the same practical sample size pattern, then scores all test rows.
final_idx, _ = train_test_split(
    np.arange(len(full_train_hist)),
    train_size=min(180000, len(full_train_hist)),
    stratify=full_train_hist['is_click'],
    random_state=RANDOM_STATE,
)
final_train_sample = full_train_hist.iloc[final_idx].copy()
final_pipe.fit(final_train_sample[full_features], final_train_sample['is_click'])
test_pred = final_pipe.predict_proba(test_final[full_features])[:, 1]
submission = pd.DataFrame({
    'session_id': test['session_id'],
    'click_probability': test_pred,
    'predicted_click': (test_pred >= best_threshold).astype(int),
})
submission.to_csv(OUTPUT_DIR/'ad_click_prediction_test_predictions.csv', index=False)
print(submission.head())

artifacts = {
    'train_shape': train.shape,
    'test_shape': test.shape,
    'click_rate': float(train['is_click'].mean()),
    'temporal_cutoff': str(cutoff.date()),
    'metrics': metrics.to_dict('records'),
    'ablation': ablation.to_dict('records'),
    'best_model': best_name,
    'best_threshold': best_threshold,
    'smote_comparison': smote_comparison,
    'weekend': weekend.to_dict('records'),
    'weekend_uplift_pp': weekend_uplift_pp,
    'weekend_relative_uplift_pct': weekend_relative_uplift,
    'weekend_p_value': float(p_value),
    'product': product.to_dict('records'),
    'profile': profile.to_dict('records'),
    'feature_importance': feature_importance.to_dict('records'),
    'business_mapping': business_mapping.to_dict('records'),
    'performance_note': 'Moderate discrimination; use as ranking/targeting aid, not a standalone automated decision system.'
}
(OUTPUT_DIR/'ctr_artifacts.json').write_text(json.dumps(artifacts, indent=2, default=str))
print('Artifacts written to outputs/ and assets/images/.')


Train: (463291, 15) Test: (128858, 14)
Click rate: 6.76 %


Temporal holdout cutoff: 2017-07-07
Dev: (391825, 24) Valid: (71466, 24) Valid click rate: 0.0616
Random stratified split is not primary because future ad-serving behavior should not leak backward into training.


Model training sample: (150000, 57) Full validation: (71466, 57)


,model,threshold,roc_auc,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,Global CTR probability baseline,0.06873,0.5,0.061582,0.116019,0.061582,1.0,0,67065,0,4401
1,Always non-click baseline,0.50000,0.5,0.061582,0.000000,0.000000,0.0,67065,0,4401,0


,model,threshold,roc_auc,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,Baseline raw + temporal - tuned threshold,0.476203,0.549089,0.070345,0.125289,0.070737,0.547603,35405,31660,1991,2410
1,+ Interaction history counts - tuned threshold,0.476203,0.590224,0.079216,0.137880,0.079206,0.531925,39850,27215,2060,2341
2,+ Leakage-safe historical CTR - tuned threshold,0.488101,0.597911,0.081791,0.143632,0.085082,0.460577,45268,21797,2374,2027


...

,product,clicks,impressions,ctr,expected_clicks_per_100k_impressions
9,J,899,9698,0.092700,9269.952568
3,D,2949,41064,0.071815,7181.472823
7,H,7654,109574,0.069852,6985.233723
2,C,11306,163501,0.069149,6914.942416
4,E,1474,21452,0.068712,6871.154205
8,I,4079,63711,0.064023,6402.348103
0,A,953,15391,0.061919,6191.930349
1,B,1238,22479,0.055074,5507.362427
5,F,344,7007,0.049094,4909.376338
6,G,435,9414,0.046208,4620.777565


,gender,age_level,city_development_index,clicks,impressions,ctr,ctr_ci_low,ctr_ci_high
51,Male,5.0,4.0,255,2945,0.086587,0.076961,0.097291
23,Female,5.0,4.0,100,1181,0.084674,0.070112,0.101930
33,Male,1.0,2.0,605,7922,0.076370,0.070724,0.082426
21,Female,5.0,2.0,208,2752,0.075581,0.066287,0.086059
37,Male,2.0,2.0,3736,50409,0.074114,0.071859,0.076433
34,Male,1.0,3.0,607,8291,0.073212,0.067800,0.079019
35,Male,1.0,4.0,306,4252,0.071966,0.064579,0.080126
50,Male,5.0,3.0,349,4909,0.071094,0.064236,0.078623
38,Male,2.0,3.0,1927,27671,0.069640,0.066700,0.072699
49,Male,5.0,2.0,493,7112,0.069319,0.063646,0.075458


,feature,importance
30,user_product_hist_ctr,0.011554
21,webpage_product_hist_clicks,0.006474
19,user_product_hist_clicks,0.005246
17,campaign_product_hist_impressions,0.003157
16,user_product_hist_impressions,0.003048
32,webpage_product_hist_ctr,0.002357
31,campaign_product_hist_ctr,0.002133
20,campaign_product_hist_clicks,0.001932
23,campaign_id_hist_ctr,0.001535
10,var_1,0.000852


,Model insight,Business action
0,High product CTR,Increase exposure and reserve more inventory
1,High webpage CTR,Prioritize placement and monitor page quality
2,High user-product historical CTR,Personalize ad selection and bid higher
3,Weekend uplift,Test weekend bid modifier before permanent ado...
4,High predicted probability,Prioritize impression under budget constraints
5,Low CTR product,Refresh creative or reduce allocation
6,Model drift,Retrain and recalibrate thresholds


   session_id  click_probability  predicted_click
0      411705           0.469205                0
1      208263           0.203322                0
2      239450           0.186125                0
3      547761           0.267432                0
4      574275           0.472063                0
Artifacts written to outputs/ and assets/images/.


## 8. Limitations and Future Improvements

- Validation metrics show moderate discrimination, not high accuracy.
- Add more behavioral history, creative metadata, bid price, placement quality, and recency features if available.
- Use SMOTENC only with a categorical-aware implementation before one-hot encoding.
- Add probability calibration and live A/B testing before production deployment.
- Monitor drift, segment stability, fairness/privacy constraints, and threshold performance over time.